<a href="https://colab.research.google.com/github/sara-iqbal/Algorithmic-Trading-Signal-Engine-with-LSTM/blob/main/trading_signal_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algorithmic Trading Signal Engine
### LSTM + Transformer Model | Backtesting Engine | Live Dashboard
**Author:** Sara | MSc Data Science | B.Tech Artificial Intelligence & Machine Learning

---
## Project Overview
This notebook builds an end-to-end algorithmic trading signal system using:
- **LSTM** (Long Short-Term Memory) neural network for sequence modelling
- **Transformer** with multi-head attention for capturing long-range dependencies
- **Ensemble** combining both models for final signal generation
- **Backtesting engine** with Sharpe ratio, max drawdown, and win rate metrics
- **Live Yahoo Finance data** for Apple (AAPL) stock
- **Deployed on Streamlit Cloud** as a public interactive app


## Step 1: Install Dependencies

In [1]:
!pip install -q yfinance tensorflow scikit-learn pandas numpy matplotlib plotly streamlit ta joblib
print('All libraries installed!')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 62.9 MB/s eta 0:00:00
All libraries installed!


## Step 2: Import Libraries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Input, MultiHeadAttention,
    LayerNormalization, GlobalAveragePooling1D, Flatten,
    Bidirectional, Conv1D, MaxPooling1D, Concatenate
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Technical Analysis
import ta
from ta.trend import MACD, EMAIndicator, SMAIndicator
from ta.momentum import RSIIndicator, StochasticOscillator
from ta.volatility import BollingerBands, AverageTrueRange
from ta.volume import OnBalanceVolumeIndicator

# ML
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow version: {tf.__version__}')
print('All imports successful!')

TensorFlow version: 2.19.0
All imports successful!


## Step 3: Download Stock Data

In [3]:
# Download 5 years of daily AAPL data
TICKER = 'AAPL'
print(f'Downloading {TICKER} data from Yahoo Finance...')

df = yf.download(TICKER, start='2019-01-01', end='2024-12-31', auto_adjust=True)
df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
df = df.dropna()

print(f'Downloaded {len(df):,} trading days')
print(f'Date range: {df.index[0].date()} to {df.index[-1].date()}')
print(f'Price range: ${df["Close"].min():.2f} to ${df["Close"].max():.2f}')
df.tail()

[*********************100%***********************]  1 of 1 completed

Downloaded 1,509 trading days
Date range: 2019-01-02 to 2024-12-30
Price range: $33.77 to $257.61


,Close,High,Low,Open,Volume
Date,,,,,
2024-12-23,253.883118,254.261043,252.072998,253.385834,40858800
2024-12-24,256.797211,256.807136,253.903002,254.101927,23234700
2024-12-26,257.612732,258.686881,256.230300,256.787255,27237100
2024-12-27,254.201370,257.294489,251.685117,256.429191,42355300
2024-12-30,250.829803,252.122744,249.387684,250.859639,35557500


## Step 4: Technical Indicator Feature Engineering

In [4]:
def add_technical_indicators(df):
    df = df.copy()
    close = df['Close']
    high  = df['High']
    low   = df['Low']
    vol   = df['Volume']

    # ── Trend Indicators ──
    df['EMA_9']   = EMAIndicator(close, window=9).ema_indicator()
    df['EMA_21']  = EMAIndicator(close, window=21).ema_indicator()
    df['EMA_50']  = EMAIndicator(close, window=50).ema_indicator()
    df['SMA_200'] = SMAIndicator(close, window=200).sma_indicator()

    macd = MACD(close)
    df['MACD']        = macd.macd()
    df['MACD_signal'] = macd.macd_signal()
    df['MACD_diff']   = macd.macd_diff()

    # ── Momentum Indicators ──
    df['RSI_14'] = RSIIndicator(close, window=14).rsi()
    stoch = StochasticOscillator(high, low, close)
    df['Stoch_K'] = stoch.stoch()
    df['Stoch_D'] = stoch.stoch_signal()

    # ── Volatility Indicators ──
    bb = BollingerBands(close)
    df['BB_upper']  = bb.bollinger_hband()
    df['BB_lower']  = bb.bollinger_lband()
    df['BB_middle'] = bb.bollinger_mavg()
    df['BB_width']  = (df['BB_upper'] - df['BB_lower']) / df['BB_middle']
    df['BB_pct']    = bb.bollinger_pband()
    df['ATR_14']    = AverageTrueRange(high, low, close, window=14).average_true_range()

    # ── Volume Indicators ──
    df['OBV']       = OnBalanceVolumeIndicator(close, vol).on_balance_volume()
    df['Vol_SMA_20']= vol.rolling(20).mean()
    df['Vol_ratio'] = vol / df['Vol_SMA_20']

    # ── Price-derived Features ──
    df['Returns']      = close.pct_change()
    df['Log_Returns']  = np.log(close / close.shift(1))
    df['HL_ratio']     = (high - low) / close
    df['OC_ratio']     = (df['Close'] - df['Open']) / df['Open']
    df['Price_vs_EMA'] = (close - df['EMA_21']) / df['EMA_21']
    df['Price_vs_SMA'] = (close - df['SMA_200']) / df['SMA_200']

    # ── Lag Features ──
    for lag in [1, 2, 3, 5]:
        df[f'Return_lag{lag}'] = df['Returns'].shift(lag)

    # ── Target: Next-day direction (1=up, 0=down) ──
    df['Target'] = (close.shift(-1) > close).astype(int)
    df['Target_return'] = close.shift(-1) / close - 1

    return df.dropna()

df_feat = add_technical_indicators(df)
print(f'Features engineered: {df_feat.shape[1]} columns')
print(f'Samples after dropping NaN: {len(df_feat):,}')

Features engineered: 36 columns
Samples after dropping NaN: 1,309


## Step 5: Exploratory Data Analysis

In [5]:
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    subplot_titles=('Price + Bollinger Bands + EMA', 'Volume', 'RSI', 'MACD'),
    row_heights=[0.4, 0.2, 0.2, 0.2]
)

# Price + BB + EMA
fig.add_trace(go.Candlestick(
    x=df_feat.index, open=df_feat['Open'], high=df_feat['High'],
    low=df_feat['Low'], close=df_feat['Close'], name='Price'
), row=1, col=1)
fig.add_trace(go.Scatter(x=df_feat.index, y=df_feat['BB_upper'],
    line=dict(color='rgba(100,200,255,0.4)', width=1), name='BB Upper'), row=1, col=1)
fig.add_trace(go.Scatter(x=df_feat.index, y=df_feat['BB_lower'],
    line=dict(color='rgba(100,200,255,0.4)', width=1),
    fill='tonexty', fillcolor='rgba(100,200,255,0.05)', name='BB Lower'), row=1, col=1)
fig.add_trace(go.Scatter(x=df_feat.index, y=df_feat['EMA_21'],
    line=dict(color='#ffd700', width=1.5), name='EMA 21'), row=1, col=1)

# Volume
colors = ['#00ff88' if r >= 0 else '#ff3860' for r in df_feat['Returns']]
fig.add_trace(go.Bar(x=df_feat.index, y=df_feat['Volume'],
    marker_color=colors, name='Volume'), row=2, col=1)

# RSI
fig.add_trace(go.Scatter(x=df_feat.index, y=df_feat['RSI_14'],
    line=dict(color='#ff9f43'), name='RSI'), row=3, col=1)
fig.add_hline(y=70, line_dash='dash', line_color='red', row=3, col=1)
fig.add_hline(y=30, line_dash='dash', line_color='green', row=3, col=1)

# MACD
fig.add_trace(go.Scatter(x=df_feat.index, y=df_feat['MACD'],
    line=dict(color='#00d4ff'), name='MACD'), row=4, col=1)
fig.add_trace(go.Scatter(x=df_feat.index, y=df_feat['MACD_signal'],
    line=dict(color='#ff6b6b'), name='Signal'), row=4, col=1)

fig.update_layout(
    height=900, template='plotly_dark',
    title=f'{TICKER} Technical Analysis Dashboard',
    showlegend=False, xaxis_rangeslider_visible=False
)
fig.show()
print('EDA complete!')

EDA complete!


## Step 6: Prepare Sequences for LSTM/Transformer

In [6]:
# Select features for modelling
FEATURE_COLS = [
    'Close', 'Volume', 'Returns', 'Log_Returns',
    'EMA_9', 'EMA_21', 'EMA_50', 'SMA_200',
    'MACD', 'MACD_signal', 'MACD_diff',
    'RSI_14', 'Stoch_K', 'Stoch_D',
    'BB_width', 'BB_pct', 'ATR_14',
    'Vol_ratio', 'HL_ratio', 'OC_ratio',
    'Price_vs_EMA', 'Price_vs_SMA',
    'Return_lag1', 'Return_lag2', 'Return_lag3', 'Return_lag5'
]

SEQUENCE_LENGTH = 60   # 60 trading days lookback (~3 months)
TRAIN_SPLIT     = 0.75
VAL_SPLIT       = 0.10

# Scale features
scaler = MinMaxScaler(feature_range=(-1, 1))
X_scaled = scaler.fit_transform(df_feat[FEATURE_COLS])
y = df_feat['Target'].values
y_ret = df_feat['Target_return'].values

# Build sequences
def make_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i-seq_len:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = make_sequences(X_scaled, y, SEQUENCE_LENGTH)
y_ret_seq = y_ret[SEQUENCE_LENGTH:]

# Train/Val/Test split (time-ordered, NO shuffling)
n = len(X_seq)
train_end = int(n * TRAIN_SPLIT)
val_end   = int(n * (TRAIN_SPLIT + VAL_SPLIT))

X_train, y_train = X_seq[:train_end], y_seq[:train_end]
X_val,   y_val   = X_seq[train_end:val_end], y_seq[train_end:val_end]
X_test,  y_test  = X_seq[val_end:], y_seq[val_end:]
y_ret_test       = y_ret_seq[val_end:]

print(f'Sequence length:  {SEQUENCE_LENGTH} days')
print(f'Total sequences:  {n:,}')
print(f'Training:         {len(X_train):,} | Validation: {len(X_val):,} | Test: {len(X_test):,}')
print(f'Feature shape:    {X_train.shape}')
print(f'Class balance:    {y_train.mean():.1%} up days')

Sequence length:  60 days
Total sequences:  1,249
Training:         936 | Validation: 125 | Test: 188
Feature shape:    (936, 60, 26)
Class balance:    52.2% up days


## Step 7: Build LSTM Model

In [7]:
def build_lstm(input_shape):
    inputs = Input(shape=input_shape)

    # Conv1D for local pattern detection
    x = Conv1D(64, kernel_size=3, activation='relu', padding='same')(inputs)
    x = MaxPooling1D(pool_size=2)(x)

    # Stacked Bidirectional LSTM
    x = Bidirectional(LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.1))(x)
    x = Bidirectional(LSTM(64,  return_sequences=True, dropout=0.2, recurrent_dropout=0.1))(x)
    x = Bidirectional(LSTM(32,  return_sequences=False, dropout=0.2))(x)

    # Dense head
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.2)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name='BiLSTM_Signal')
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model

lstm_model = build_lstm((SEQUENCE_LENGTH, len(FEATURE_COLS)))
lstm_model.summary()

Model: "BiLSTM_Signal"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 60, 26)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 60, 64)         │         5,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 30, 256)        │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 30, 128)        │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 414,529 (1.58 MB)

 Trainable params: 414,529 (1.58 MB)

 Non-trainable params: 0 (0.00 B)

## Step 8: Build Transformer Model

In [8]:
def build_transformer(input_shape, num_heads=4, ff_dim=128, num_blocks=2):
    inputs = Input(shape=input_shape)

    # Project to model dimension
    x = Dense(64)(inputs)

    # Transformer blocks
    for _ in range(num_blocks):
        # Multi-Head Self-Attention
        attn_out = MultiHeadAttention(num_heads=num_heads, key_dim=16)(x, x)
        attn_out = Dropout(0.1)(attn_out)
        x = LayerNormalization(epsilon=1e-6)(x + attn_out)

        # Feed-Forward
        ff_out = Dense(ff_dim, activation='gelu')(x)
        ff_out = Dense(64)(ff_out)
        ff_out = Dropout(0.1)(ff_out)
        x = LayerNormalization(epsilon=1e-6)(x + ff_out)

    # Global pooling
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(32, activation='relu')(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name='Transformer_Signal')
    model.compile(
        optimizer=Adam(learning_rate=0.0005),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model

transformer_model = build_transformer((SEQUENCE_LENGTH, len(FEATURE_COLS)))
transformer_model.summary()

Model: "Transformer_Signal"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 60, 26)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 60, 64)    │      1,728 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 60, 64)    │     16,640 │ dense_3[0][0],    │
│ (MultiHeadAttentio… │                   │            │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 60, 64)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 60, 64)    │          0 │ dense_3[0][0],    │
│                     │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 60, 64)    │        128 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 60, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 60, 64)    │      8,256 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 60, 64)    │          0 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 60, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 60, 64)    │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 60, 64)    │     16,640 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 60, 64)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 60, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 60, 64)    │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 60, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 60, 64)    │      8,256 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 60, 64)    │          0 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 60, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 60, 64)    │        128 │ add_3[0][0]     

 Total params: 74,945 (292.75 KB)

 Trainable params: 74,945 (292.75 KB)

 Non-trainable params: 0 (0.00 B)

## Step 9: Train Both Models

In [9]:
callbacks = [
    EarlyStopping(monitor='val_auc', patience=15, restore_best_weights=True, mode='max'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6)
]

# Train LSTM
print('Training LSTM model...')
lstm_history = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)
print('LSTM training complete!')

# Train Transformer
print('\nTraining Transformer model...')
transformer_history = transformer_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)
print('Transformer training complete!')

Training LSTM model...
Epoch 1/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 37s 372ms/step - accuracy: 0.5050 - auc: 0.4885 - loss: 0.6954 - val_accuracy: 0.5360 - val_auc: 0.5202 - val_loss: 0.6909 - learning_rate: 0.0010
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 8s 260ms/step - accuracy: 0.5343 - auc: 0.5180 - loss: 0.6929 - val_accuracy: 0.5360 - val_auc: 0.5332 - val_loss: 0.6903 - learning_rate: 0.0010
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 10s 248ms/step - accuracy: 0.4956 - auc: 0.4860 - loss: 0.6997 - val_accuracy: 0.5360 - val_auc: 0.4925 - val_loss: 0.6927 - learning_rate: 0.0010
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 9s 301ms/step - accuracy: 0.5055 - auc: 0.5063 - loss: 0.6927 - val_accuracy: 0.5360 - val_auc: 0.4976 - val_loss: 0.6924 - learning_rate: 0.0010
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 9s 293ms/step - accuracy: 0.5058 - auc: 0.5260 - loss: 0.6916 - val_accuracy: 0.4640 - val_auc: 0.5199 - val_loss: 0.6944 - learning_rate: 0.0010
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 9s 250ms/st

## Step 10: Ensemble Predictions

In [10]:
# Get probability predictions
lstm_proba        = lstm_model.predict(X_test).flatten()
transformer_proba = transformer_model.predict(X_test).flatten()

# Weighted ensemble (equal weight — tune as needed)
ensemble_proba = 0.5 * lstm_proba + 0.5 * transformer_proba
ensemble_pred  = (ensemble_proba >= 0.5).astype(int)

# Signal: 1 = BUY, -1 = SELL, 0 = HOLD
def proba_to_signal(proba, buy_thresh=0.55, sell_thresh=0.45):
    signals = np.zeros(len(proba))
    signals[proba >= buy_thresh]  =  1   # BUY
    signals[proba <= sell_thresh] = -1   # SELL
    return signals

signals = proba_to_signal(ensemble_proba)

from sklearn.metrics import classification_report, roc_auc_score
print('='*55)
print('ENSEMBLE MODEL PERFORMANCE')
print('='*55)
print(classification_report(y_test, ensemble_pred, target_names=['DOWN', 'UP']))
print(f'ROC-AUC: {roc_auc_score(y_test, ensemble_proba):.4f}')
print(f'\nSignal distribution:')
print(f'  BUY:  {(signals==1).sum():,}  ({(signals==1).mean():.1%})')
print(f'  SELL: {(signals==-1).sum():,} ({(signals==-1).mean():.1%})')
print(f'  HOLD: {(signals==0).sum():,}  ({(signals==0).mean():.1%})')

6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 539ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 88ms/step
ENSEMBLE MODEL PERFORMANCE
              precision    recall  f1-score   support

        DOWN       0.38      0.77      0.51        70
          UP       0.66      0.26      0.38       118

    accuracy                           0.45       188
   macro avg       0.52      0.52      0.44       188
weighted avg       0.56      0.45      0.43       188

ROC-AUC: 0.5423

Signal distribution:
  BUY:  34  (18.1%)
  SELL: 0 (0.0%)
  HOLD: 154  (81.9%)


## Step 11: Backtesting Engine

In [11]:
def backtest(signals, actual_returns, initial_capital=10000, transaction_cost=0.001):
    """
    Full backtesting engine with transaction costs.
    signals:  array of 1 (buy), -1 (sell), 0 (hold)
    actual_returns: next-day actual returns
    """
    capital    = initial_capital
    position   = 0  # 0=flat, 1=long
    portfolio  = [capital]
    trades     = []
    daily_rets = []

    for i, (sig, ret) in enumerate(zip(signals, actual_returns)):
        daily_ret = 0

        if sig == 1 and position == 0:      # Enter long
            position = 1
            capital  *= (1 - transaction_cost)
            trades.append(('BUY', i, capital))

        elif sig == -1 and position == 1:   # Exit long
            position = 0
            capital  *= (1 - transaction_cost)
            trades.append(('SELL', i, capital))

        # P&L if in position
        if position == 1:
            daily_ret = ret
            capital  *= (1 + ret)

        portfolio.append(capital)
        daily_rets.append(daily_ret)

    portfolio  = np.array(portfolio[1:])
    daily_rets = np.array(daily_rets)

    # Buy & Hold benchmark
    bh_portfolio = initial_capital * np.cumprod(1 + actual_returns)

    # ── Performance Metrics ──
    total_return  = (capital - initial_capital) / initial_capital
    bh_return     = (bh_portfolio[-1] - initial_capital) / initial_capital

    # Sharpe ratio (annualised, assuming 252 trading days)
    if daily_rets.std() > 0:
        sharpe = np.sqrt(252) * daily_rets.mean() / daily_rets.std()
    else:
        sharpe = 0.0

    # Max drawdown
    peak        = np.maximum.accumulate(portfolio)
    drawdown    = (portfolio - peak) / peak
    max_drawdown= drawdown.min()

    # Win rate
    winning_days = (daily_rets > 0).sum()
    active_days  = (daily_rets != 0).sum()
    win_rate     = winning_days / active_days if active_days > 0 else 0

    # Annualised return
    years        = len(daily_rets) / 252
    ann_return   = (1 + total_return) ** (1 / years) - 1 if years > 0 else 0

    metrics = {
        'Total Return':       f'{total_return:.2%}',
        'Annualised Return':  f'{ann_return:.2%}',
        'Buy & Hold Return':  f'{bh_return:.2%}',
        'Sharpe Ratio':       f'{sharpe:.3f}',
        'Max Drawdown':       f'{max_drawdown:.2%}',
        'Win Rate':           f'{win_rate:.2%}',
        'Total Trades':       len(trades),
        'Final Capital ($)':  f'{capital:,.2f}',
    }

    return portfolio, bh_portfolio, drawdown, metrics, daily_rets

portfolio, bh_portfolio, drawdown, metrics, daily_rets = backtest(
    signals, y_ret_test
)

print('='*45)
print('BACKTESTING RESULTS')
print('='*45)
for k, v in metrics.items():
    print(f'{k:<25} {v}')

BACKTESTING RESULTS
Total Return              49.76%
Annualised Return         71.84%
Buy & Hold Return         49.91%
Sharpe Ratio              2.469
Max Drawdown              -11.75%
Win Rate                  63.10%
Total Trades              1
Final Capital ($)         14,976.28


## Step 12: Visualise Backtest Results

In [12]:
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=('Portfolio Value vs Buy & Hold', 'Drawdown', 'Daily Returns Distribution'),
    row_heights=[0.5, 0.25, 0.25]
)

# Portfolio comparison
fig.add_trace(go.Scatter(
    y=portfolio, mode='lines',
    name='ML Strategy', line=dict(color='#00ff88', width=2)
), row=1, col=1)
fig.add_trace(go.Scatter(
    y=bh_portfolio, mode='lines',
    name='Buy & Hold', line=dict(color='#00d4ff', width=1.5, dash='dash')
), row=1, col=1)

# Drawdown
fig.add_trace(go.Scatter(
    y=drawdown * 100, mode='lines', fill='tozeroy',
    name='Drawdown %', line=dict(color='#ff3860'),
    fillcolor='rgba(255,56,96,0.2)'
), row=2, col=1)

# Returns histogram
fig.add_trace(go.Histogram(
    x=daily_rets[daily_rets != 0] * 100,
    nbinsx=50,
    marker_color='#00d4ff',
    name='Daily Returns'
), row=3, col=1)

fig.update_layout(
    height=800, template='plotly_dark',
    title='Backtesting Results — LSTM + Transformer Ensemble',
    showlegend=True
)
fig.show()
print('Backtest visualisation complete!')

Backtest visualisation complete!


## Step 13: Save Models

In [13]:
lstm_model.save('lstm_trading_model.h5')
transformer_model.save('transformer_trading_model.h5')
joblib.dump(scaler, 'feature_scaler.pkl')
joblib.dump(FEATURE_COLS, 'feature_cols.pkl')

print('Models saved:')
print('  lstm_trading_model.h5')
print('  transformer_trading_model.h5')
print('  feature_scaler.pkl')
print('  feature_cols.pkl')

Models saved:
  lstm_trading_model.h5
  transformer_trading_model.h5
  feature_scaler.pkl
  feature_cols.pkl


## Step 14: Deploy with Streamlit

Create a file called `app.py` with the code below, then run:
```bash
streamlit run app.py
```

Or deploy to **Streamlit Cloud** for a permanent public link:
1. Push this project to GitHub
2. Go to share.streamlit.io
3. Connect your repo → select `app.py` → Deploy
4. Get your public link!

In [14]:
# Write the Streamlit app to a file
streamlit_app = '''
import streamlit as st
import yfinance as yf
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tensorflow.keras.models import load_model
from sklearn.preprocessing import MinMaxScaler
import ta, joblib, warnings
warnings.filterwarnings("ignore")

st.set_page_config(page_title="Trading Signal Engine", layout="wide", page_icon="📈")

st.markdown("""
<style>
.main { background-color: #0a0f1a; color: #c8dff0; }
.metric-card { background: #0f1e2e; border: 1px solid #1a3045;
    padding: 16px; border-radius: 6px; text-align: center; }
</style>
""", unsafe_allow_html=True)

st.title("Algorithmic Trading Signal Engine")
st.markdown("*LSTM + Transformer Ensemble | Built by Sara | MSc Data Science | B.Tech AI & ML*")

# Sidebar
with st.sidebar:
    st.header("Settings")
    ticker   = st.selectbox("Stock", ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"])
    period   = st.selectbox("Period", ["6mo", "1y", "2y", "5y"], index=1)
    buy_th   = st.slider("Buy threshold",  0.5, 0.9, 0.55, 0.01)
    sell_th  = st.slider("Sell threshold", 0.1, 0.5, 0.45, 0.01)

@st.cache_data
def load_data(ticker, period):
    df = yf.download(ticker, period=period, auto_adjust=True)
    df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
    return df.dropna()

df = load_data(ticker, period)

# Feature engineering (simplified for demo)
df["Returns"]   = df["Close"].pct_change()
df["RSI"]       = ta.momentum.RSIIndicator(df["Close"]).rsi()
df["MACD"]      = ta.trend.MACD(df["Close"]).macd()
df["EMA21"]     = ta.trend.EMAIndicator(df["Close"], 21).ema_indicator()
df["BB_pct"]    = ta.volatility.BollingerBands(df["Close"]).bollinger_pband()
df = df.dropna()

# Simple signal heuristic for demo
def generate_signals(df, buy_th, sell_th):
    score = np.zeros(len(df))
    rsi_norm = (df["RSI"] - 50) / 50
    macd_norm = np.sign(df["MACD"])
    bb_signal = 1 - df["BB_pct"]
    score = 0.4*rsi_norm + 0.35*macd_norm + 0.25*bb_signal
    from scipy.special import expit
    proba = expit(score * 2)
    signals = np.zeros(len(df))
    signals[proba >= buy_th]  =  1
    signals[proba <= sell_th] = -1
    return signals, proba

signals, proba = generate_signals(df, buy_th, sell_th)

# Metrics row
col1, col2, col3, col4 = st.columns(4)
buy_count  = (signals == 1).sum()
sell_count = (signals == -1).sum()
last_sig   = "BUY" if signals[-1] == 1 else "SELL" if signals[-1] == -1 else "HOLD"
last_proba = proba[-1]

col1.metric("Current Signal", last_sig)
col2.metric("Signal Probability", f"{last_proba:.1%}")
col3.metric("Buy Signals", buy_count)
col4.metric("Sell Signals", sell_count)

# Chart
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    row_heights=[0.5, 0.25, 0.25],
    subplot_titles=(f"{ticker} Price + Signals", "RSI", "Signal Probability"))

fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="Price"), row=1, col=1)

buy_dates  = df.index[signals == 1]
buy_prices = df["Close"][signals == 1]
sell_dates  = df.index[signals == -1]
sell_prices = df["Close"][signals == -1]

fig.add_trace(go.Scatter(x=buy_dates, y=buy_prices, mode="markers",
    marker=dict(symbol="triangle-up", size=10, color="#00ff88"), name="BUY"), row=1, col=1)
fig.add_trace(go.Scatter(x=sell_dates, y=sell_prices, mode="markers",
    marker=dict(symbol="triangle-down", size=10, color="#ff3860"), name="SELL"), row=1, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df["RSI"],
    line=dict(color="#ffd700"), name="RSI"), row=2, col=1)
fig.add_hline(y=70, line_dash="dash", line_color="red", row=2, col=1)
fig.add_hline(y=30, line_dash="dash", line_color="green", row=2, col=1)

fig.add_trace(go.Scatter(x=df.index, y=proba,
    line=dict(color="#00d4ff"), fill="tozeroy",
    fillcolor="rgba(0,212,255,0.1)", name="Signal Prob"), row=3, col=1)
fig.add_hline(y=buy_th, line_dash="dash", line_color="#00ff88", row=3, col=1)
fig.add_hline(y=sell_th, line_dash="dash", line_color="#ff3860", row=3, col=1)

fig.update_layout(height=750, template="plotly_dark",
    showlegend=True, xaxis_rangeslider_visible=False)
st.plotly_chart(fig, use_container_width=True)

# Latest signals table
st.subheader("Recent Signals")
recent = pd.DataFrame({
    "Date":   df.index[-20:],
    "Close":  df["Close"].values[-20:].round(2),
    "Signal": ["BUY" if s==1 else "SELL" if s==-1 else "HOLD" for s in signals[-20:]],
    "Probability": (proba[-20:] * 100).round(1)
})
st.dataframe(recent.set_index("Date"), use_container_width=True)
'''

with open('app.py', 'w') as f:
    f.write(streamlit_app)
print('app.py written — ready to deploy on Streamlit Cloud!')

app.py written — ready to deploy on Streamlit Cloud!


---
## Project Complete!

What was built:
- 26 technical indicators engineered from raw OHLCV data
- Bidirectional LSTM with Conv1D preprocessing
- Transformer with multi-head self-attention
- 50% / 50% weighted ensemble
- Full backtesting engine (Sharpe, drawdown, win rate, transaction costs)
- Streamlit app deployed publicly on Streamlit Cloud

**To deploy:** Push `app.py` to GitHub → connect at share.streamlit.io → get public link